# Zielsetzung

Ziel dieser arbeit ist es den Zustand einer Maschine innerhalb einer Produktionsanlage zu überwachen. Dafür soll anhand von verschiedenen Sensor- und Kontextinformationenen abgeschätzt werden, ob ein technischer Defekt vorliegt, wie groß das Risiko eines kurzfristigen ausfalls ist und welche Auswirkungen ein ungeplanter stillstand auf die Produktion hätte.

Diese Informationen sollen anschließend helfen eine Entscheidung zutreffen, ob die Maschine:
* weiter betrieben werden kann
* zeitnah gewartet werden soll
* sofort gestoppt werden sollte

## (i) Technischer Zustand der Maschine
Die Wahrscheinlichkeit eines technischen Defekts soll modelliert werden.

Sensorinformationen:
* Vibrationsniveau {niedrig, mittel, hoch}
* Temperatur {normal, erhöht, kritisch}

Aus diesen beiden Grßen soll eine gemeinsame Zufallsvariable 
* Sensorzustand {unauffällig, auffällig, kritisch}
modelliert werden.
Dabei soll ein hohes Vibrationsniveau und/oder eine erhöhte Temperatur die Wahrscheinlichkeit eins auffälligen oder kritischen Sensorzustands erhöhen.

Der Defektzustand {kein Defekt, leichter Defekt, kritischer Defekt} soll hingegen vom Sensorzustand und dem Maschinenalter {neu,mittel, alt}
 abhängen. 



## (ii) Risiko eines kurzfristigen Maschinenausfalls

## (iii) Auswirkungen eins Maschinenstillstands

# (i) Technischer Zustand der Maschine 

| Variable | Mögliche Zustände | 
| :--- | :---: | 
| Vibrationsniveau | niedrig, mittel, hoch | 
| Temperatur | normal, erhöht, kritisch | 
| Sensorzustand  | unauffällig, auffällig, kritisch | 
| Maschinenalter | neu, mittel, alt | 
| Defektzustand | kein Defekt, leichter Defekt, kritischer Defekt | 

                   
## mögliche 
| Vibration | Temperatur | P(unauffällig) | P(auffällig) | P(kritisch) |
| niedrig | normal | 0,90 | 0,09 | 0,01 |
| niedrig | erhöht | 0,55 | 0,44 | 0,05 | 
| niedrig | kritisch | 0,10 | 0,35 | 0,55 | 
| hoch | normal | 0,15 | 0,60 | 0,25 | 
| hoch | kritisch | 0,01 | 0,09 | 0,90 | 

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ("Vibrationsniveau", "Sensorzustand"),
    ("Temperatur", "Sensorzustand"),
    ("Sensorzustand", "Defektzustand"),
    ("Maschinenalter", "Defektzustand")
])

# Priorwahrscheinlichkeiten 
cpd_alter = TabularCPD(
    variable="Maschinenalter",
    variable_card=3,
    values=[
        [0.30], # neu
        [0.45], # mittel
        [0.25]  # alt
    ],
    state_names={
        "Maschinenalter": ["neu", "mittel", "alt"]
    }
)

cpd_vibration = TabularCPD(
    variable="Vibrationsniveau",
    variable_card=3,
    values=[
        [0.60], # niedrig
        [0.30], # mittel
        [0.10]  # hoch
    ],
    state_names={
        "Vibrationsniveau": ["niedrig", "mittel", "hoch"]
    }
)

cpd_temperatur = TabularCPD(
    variable="Temperatur",
    variable_card=3,
    values=[
        [0.70], # normal
        [0.25], # erhöht 
        [0.05], # kritisch
    ],
    state_names={
        "Temperatur": ["normal", "erhöht", "kritisch"]
    }
)

cpd_sensor = TabularCPD(
    variable="Sensorzustand",
    variable_card=3,
    values=[
        # niedrig               mittel                hoch
        # normal erhöht krit. | normal erhöht krit. | normal erhöht krit.
        [0.90, 0.55, 0.10,      0.65, 0.25, 0.03,     0.15, 0.03, 0.01], # unauffällig
        [0.09, 0.40, 0.35,      0.30, 0.60, 0.22,     0.60, 0.32, 0.09], # auffällig
        [0.01, 0.05, 0.55,      0.05, 0.15, 0.75,     0.25, 0.65, 0.90]  # kritisch
    ],
    evidence=["Vibrationsniveau", "Temperatur"],
    evidence_card=[3, 3],
    state_names={
        "Sensorzustand": [
            "unauffällig", "auffällig", "kritisch"
        ],
        "Vibrationsniveau": [
            "niedrig","mittel", "hoch"
        ],
        "Temperatur": [
            "normal", "erhöht", "kritisch"
        ]
    }
)

cpd_defekt = TabularCPD(
    variable="Defektzustand",
    variable_card=3,
    values=[
        # unauffällig          auffällig           kritisch
        # neu   mittel alt  | neu   mittel alt  | neu   mittel alt
        [0.95, 0.85, 0.70,   0.60, 0.40, 0.20,   0.20, 0.10, 0.02], # kein Defekt
        [0.04, 0.12, 0.22,   0.35, 0.45, 0.50,   0.50, 0.40, 0.18], # leichter Defekt
        [0.01, 0.03, 0.08,   0.05, 0.15, 0.30,   0.30, 0.50, 0.80]  # kritischer Defekt

    ],
    evidence=["Sensorzustand", "Maschinenalter"],
    evidence_card=[3,3],
    state_names={
        "Defektzustand": [
            "kein Defekt", "leichter Defekt", "kritischer Defekt"
        ],
        "Sensorzustand": [
            "unauffällig", "auffällig", "kritisch"
        ],
        "Maschinenalter": [
            "neu", "mittel", "alt"
        ]
    }
)






model.add_cpds(
    cpd_alter,
    cpd_vibration,
    cpd_temperatur,
    cpd_sensor,
    cpd_defekt
)

print(model.check_model())

inference = VariableElimination(model)

result = inference.query(
    variables=["Defektzustand"],
    evidence={
        "Vibrationsniveau": "hoch",
        "Temperatur": "kritisch",
        "Maschinenalter": "alt"
    }
)

print(result)

ValueError: No CPD associated with Vibrationsniveau